<a href="https://colab.research.google.com/github/saikirankesoju/Reinforcement_Learning_LAB/blob/main/RL_ASSI_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Assignment 1: Bellman Optimality and Value Iteration
# Deterministic 2x2 Grid World

# 1. Define MDP parameters
gamma = 0.9
GOAL_REWARD = 10
NORMAL_REWARD = -1
WALL_REWARD = -1
theta = 0.000001

states = ["S0", "S1", "G"]
actions = ["Up", "Down", "Left", "Right"]
terminal_state = "G"

# 2. Define deterministic transitions
# Each transition contains (next_state, immediate_reward)

transitions = {
    "S0": {
        "Up": ("S0", NORMAL_REWARD),
        "Down": ("S1", NORMAL_REWARD),
        "Left": ("S0", NORMAL_REWARD),
        "Right": ("S0", WALL_REWARD)
    },
    "S1": {
        "Up": ("S0", NORMAL_REWARD),
        "Down": ("S1", NORMAL_REWARD),
        "Left": ("S1", NORMAL_REWARD),
        "Right": ("G", GOAL_REWARD)
    }
}

# 3. Display the Grid World
print("=" * 65)
print("DETERMINISTIC GRID WORLD")
print("=" * 65)
print("Grid layout:")
print("+----------------+----------------+")
print("| S0             | WALL           |")
print("| Start State    | Reward = -1    |")
print("+----------------+----------------+")
print("| S1             | G              |")
print("| Intermediate   | Goal           |")
print("+----------------+----------------+")
print("States:", states)
print("Actions:", actions)
print("Discount factor:", gamma)
print("Normal reward:", NORMAL_REWARD)
print("Wall reward:", WALL_REWARD)
print("Goal reward:", GOAL_REWARD)
print("Terminal state:", terminal_state)

# 4. Verify every state-action transition
print("\n" + "=" * 65)
print("TRANSITION VERIFICATION")
print("=" * 65)

for state in ["S0", "S1"]:
    for action in actions:
        next_state, reward = transitions[state][action]
        print(
            f"State: {state:2} | Action: {action:5} | "
            f"Next State: {next_state:2} | Reward: {reward}"
        )

        assert next_state in states

print("\nAll non-terminal state-action pairs have transitions.")

# 5. Initialize state values
V = {state: 0.0 for state in states}
V[terminal_state] = 0.0

print("\n" + "=" * 65)
print("INITIAL STATE VALUES")
print("=" * 65)

for state in states:
    print(f"V({state}) = {V[state]:.6f}")

# 6. Implement synchronous Value Iteration
print("\n" + "=" * 65)
print("VALUE ITERATION")
print("=" * 65)

iteration = 0
iteration_history = []

while True:
    iteration += 1

    # Use only previous iteration values
    old_V = V.copy()
    new_V = old_V.copy()

    for state in ["S0", "S1"]:
        action_values = []

        for action in actions:
            next_state, reward = transitions[state][action]

            # Bellman Optimality equation
            q_value = reward + gamma * old_V[next_state]
            action_values.append(q_value)

        new_V[state] = max(action_values)

    # The goal is terminal
    new_V[terminal_state] = 0.0

    # Calculate maximum change
    delta = max(abs(new_V[s] - old_V[s]) for s in states)

    V = new_V

    iteration_history.append({
        "Iteration": iteration,
        "V(S0)": V["S0"],
        "V(S1)": V["S1"],
        "V(G)": V["G"],
        "Delta": delta
    })

    print(
        f"Iteration {iteration:2}: "
        f"V(S0) = {V['S0']:10.6f}, "
        f"V(S1) = {V['S1']:10.6f}, "
        f"V(G) = {V['G']:8.6f}, "
        f"Delta = {delta:.8f}"
    )

    if delta < theta:
        break

print("\nValue Iteration converged.")
print("Convergence threshold:", theta)
print("Total iterations:", iteration)

# 7. Display final optimal state values
print("\n" + "=" * 65)
print("FINAL OPTIMAL STATE VALUES")
print("=" * 65)

for state in states:
    print(f"V*({state}) = {V[state]:.6f}")

# 8. Calculate optimal action values Q*(s, a)
print("\n" + "=" * 65)
print("OPTIMAL ACTION VALUES Q*(s, a)")
print("=" * 65)

Q = {}

for state in ["S0", "S1"]:
    Q[state] = {}

    for action in actions:
        next_state, reward = transitions[state][action]

        Q[state][action] = reward + gamma * V[next_state]

        print(
            f"Q*({state}, {action:5}) = "
            f"{Q[state][action]:10.6f}"
        )

# 9. Extract the optimal policy
print("\n" + "=" * 65)
print("OPTIMAL POLICY")
print("=" * 65)

policy = {}

for state in ["S0", "S1"]:
    best_value = max(Q[state].values())

    best_actions = [
        action for action in actions
        if abs(Q[state][action] - best_value) < 1e-9
    ]

    policy[state] = best_actions

    print(f"State {state}: Optimal action(s) = {best_actions}")
    print(f"Maximum Q-value = {best_value:.6f}")

# 10. Verify Bellman Optimality
print("\n" + "=" * 65)
print("BELLMAN OPTIMALITY VERIFICATION")
print("=" * 65)

for state in ["S0", "S1"]:
    max_q = max(Q[state].values())

    print(
        f"State: {state} | "
        f"V*({state}) = {V[state]:.6f} | "
        f"max Q*({state}, a) = {max_q:.6f}"
    )

    assert abs(V[state] - max_q) < 1e-9

print("\nVerification successful!")
print("V*(s) = max_a Q*(s, a) for every non-terminal state.")

# 11. Display iteration history as a table
print("\n" + "=" * 65)
print("ITERATION HISTORY TABLE")
print("=" * 65)

try:
    import pandas as pd

    df = pd.DataFrame(iteration_history)
    print(df.to_string(index=False))

except ImportError:
    for record in iteration_history:
        print(record)

# 12. Explain the result
print("\n" + "=" * 65)
print("RESULT EXPLANATION")
print("=" * 65)

print("1. The goal state is terminal, so V(G) = 0.")
print("2. Moving to a normal cell gives a reward of -1.")
print("3. Hitting the wall gives a reward of -1.")
print("4. Reaching the goal gives a positive reward.")
print("5. Value Iteration selects the action with the highest Q-value.")
print("6. The discount factor gamma = 0.9 reduces future rewards.")
print("7. The optimal policy maximizes the expected discounted return.")

DETERMINISTIC GRID WORLD
Grid layout:
+----------------+----------------+
| S0             | WALL           |
| Start State    | Reward = -1    |
+----------------+----------------+
| S1             | G              |
| Intermediate   | Goal           |
+----------------+----------------+
States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount factor: 0.9
Normal reward: -1
Wall reward: -1
Goal reward: 10
Terminal state: G

TRANSITION VERIFICATION
State: S0 | Action: Up    | Next State: S0 | Reward: -1
State: S0 | Action: Down  | Next State: S1 | Reward: -1
State: S0 | Action: Left  | Next State: S0 | Reward: -1
State: S0 | Action: Right | Next State: S0 | Reward: -1
State: S1 | Action: Up    | Next State: S0 | Reward: -1
State: S1 | Action: Down  | Next State: S1 | Reward: -1
State: S1 | Action: Left  | Next State: S1 | Reward: -1
State: S1 | Action: Right | Next State: G  | Reward: 10

All non-terminal state-action pairs have transitions.

INITIAL STATE VALUES
V(S0